# Árboles de Decisión para Clasificación

_Gini, entropía y comparación con la regresión logística — caso: default crediticio_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

![Aprendizaje Supervisado](assets/header.png)

## 1. Intuición

Igual que en regresión, dividimos el espacio en regiones rectangulares. La diferencia es que en cada hoja, en lugar de una media, predecimos la **clase mayoritaria** (o un vector de probabilidades estimadas como las frecuencias relativas dentro de la hoja). Más adelante visualizamos un árbol real entrenado sobre el dataset de **default crediticio**.

## 2. ¿Qué hace un "buen" split? — pureza

Queremos splits que dejen los hijos lo más **"puros"** posible: idealmente, todas las observaciones de un hijo de la misma clase. Para medir cuán impuro es un nodo se usan dos índices:

**Índice de Gini** (más usado, default en scikit-learn):

$$
G = 1 - \sum_{k=1}^{K} p_k^{2}
$$

**Entropía**:

$$
H = -\sum_{k=1}^{K} p_k \log_2 p_k
$$

donde $p_k$ es la proporción de la clase $k$ en el nodo. En un problema binario ambas alcanzan su **máximo cuando las clases están 50/50** (máxima incertidumbre) y valen **0 cuando el nodo es puro**:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = np.linspace(0.001, 0.999, 200)
gini = 1 - p**2 - (1 - p)**2
ent  = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, gini, label='Gini', lw=2)
ax.plot(p, ent,  label='Entropía', lw=2)
ax.set_xlabel('p (proporción de la clase 1)'); ax.set_ylabel('impureza')
ax.set_title('Gini y entropía en un nodo binario')
ax.legend(); ax.grid(True); plt.show()

El árbol busca el split que **más reduce la impureza** ponderando por el tamaño de los hijos. Eso se llama **ganancia de información** y es lo que decide qué variable y qué corte usar en cada nodo.

## 3. Logística vs árbol

| | Logística | Árbol |
|---|---|---|
| Frontera de decisión | lineal | escalonada (cajas) |
| No linealidades / interacciones | manuales | automáticas |
| Salida | probabilidades calibradas | proporciones por hoja |
| Interpretabilidad | coeficientes | reglas if/else |
| Sensibilidad a outliers | media | baja |
| Riesgo de overfitting | bajo (con regularización) | alto sin poda |

Visualmente, en un mismo problema 2D la logística traza una **recta** y el árbol traza **rectángulos**:

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

Xs, ys = make_moons(n_samples=400, noise=0.25, random_state=0)
modelos = [
    ('Regresión logística', LogisticRegression().fit(Xs, ys)),
    ('Árbol (max_depth=4)', DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xs, ys)),
]

xx, yy = np.meshgrid(
    np.linspace(Xs[:,0].min()-0.5, Xs[:,0].max()+0.5, 300),
    np.linspace(Xs[:,1].min()-0.5, Xs[:,1].max()+0.5, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (nombre, m) in zip(axes, modelos):
    zz = m.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['#FFE4B5', '#B0E0E6'])
    ax.scatter(Xs[:,0], Xs[:,1], c=ys, cmap='coolwarm', edgecolor='k', s=25)
    ax.set_title(nombre)
plt.tight_layout(); plt.show()

## 4. Caso práctico — Loan Default Dataset (Kaggle)

**Dataset:** https://www.kaggle.com/datasets/yasserh/loan-default-dataset  
**Archivo:** `Loan_Default.csv` (148.670 × 34).

**Estructura.** Cada fila es un préstamo hipotecario. Las columnas se pueden agrupar así:

- **Identificador / metadata**: `ID`, `year`.
- **Solicitante**: `Gender`, `age` (rango), `income`, `Credit_Score`, `credit_type` (CIB / CRIF / EXP / EQUI), `co-applicant_credit_type`.
- **Préstamo**: `loan_amount`, `loan_type`, `loan_purpose`, `loan_limit`, `Credit_Worthiness`, `term` (meses), `rate_of_interest`, `Interest_rate_spread`, `Upfront_charges`, `business_or_commercial`, `open_credit`, `interest_only`, `Neg_ammortization`, `lump_sum_payment`, `submission_of_application`, `approv_in_adv`.
- **Garantía / inmueble**: `property_value`, `LTV` (loan-to-value), `construction_type`, `occupancy_type`, `Secured_by`, `Security_Type`, `total_units`.
- **Geografía**: `Region` (North / south / central / North-East).
- **Capacidad de pago**: `dtir1` (debt-to-income ratio).
- **Target**: `Status` (1 = entró en default, 0 = pagó al día). Tasa de default ≈ 24.6%.

**Problema:** clasificación binaria — predecir si un préstamo va a entrar en default. Es un caso clásico de **scoring crediticio**: bancos y fintechs lo usan para decidir aprobaciones, tasas y reservas de capital.

**Variables que usaremos.** Tomamos un subconjunto representativo (numéricas + categóricas) y rellenamos los NaN: numéricos con la **mediana**, categóricos con el literal `'Unknown'`.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/Loan_Default.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/yasserh/loan-default-dataset '
        'y colócalo en data/Loan_Default.csv (ver README.md).'
    )

df = pd.read_csv(DATA)
print('Shape:', df.shape, '| Tasa de default:', round(df['Status'].mean(), 3))
df.head()

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report,
)

sns.set_theme(style='whitegrid')

num_cols = ['loan_amount', 'rate_of_interest', 'term', 'property_value',
            'income', 'Credit_Score', 'LTV', 'dtir1']
cat_cols = ['loan_type', 'loan_purpose', 'Credit_Worthiness',
            'business_or_commercial', 'occupancy_type', 'credit_type',
            'age', 'Region', 'Gender']

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    df[c] = df[c].fillna('Unknown')

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Status']
print('Shape de X:', X.shape)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_tr.shape, X_te.shape

In [ ]:
tree = DecisionTreeClassifier(
    criterion='gini', max_depth=4, random_state=42,
).fit(X_tr, y_tr)

y_pred  = tree.predict(X_te)
y_proba = tree.predict_proba(X_te)[:, 1]

print(f'Accuracy  : {accuracy_score(y_te, y_pred):.3f}')
print(f'Precision : {precision_score(y_te, y_pred):.3f}')
print(f'Recall    : {recall_score(y_te, y_pred):.3f}')
print(f'F1        : {f1_score(y_te, y_pred):.3f}')
print(f'ROC AUC   : {roc_auc_score(y_te, y_proba):.3f}')

print('\n', classification_report(y_te, y_pred, target_names=['Al día', 'Default']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['Al día', 'Default'],
).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Matriz de confusión — árbol')

RocCurveDisplay.from_predictions(y_te, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Curva ROC — árbol')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=['Al día', 'Default'],
    filled=True, rounded=True, ax=ax, fontsize=8,
)
ax.set_title('Árbol de clasificación sobre Loan Default (max_depth=4)')
plt.show()

In [ ]:
imp = (
    pd.Series(tree.feature_importances_, index=X.columns)
    .sort_values()
    .tail(12)
)
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, color='teal')
ax.set_title('Importancia de variables — DecisionTreeClassifier')
plt.show()

In [ ]:
# --- Comparación contra logística sobre el mismo split ---
scaler = StandardScaler().fit(X_tr)
logit = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)
log_pred  = logit.predict(scaler.transform(X_te))
log_proba = logit.predict_proba(scaler.transform(X_te))[:, 1]

comparativa = pd.DataFrame({
    'modelo':   ['Decision Tree', 'Regresión logística'],
    'accuracy': [accuracy_score(y_te, y_pred), accuracy_score(y_te, log_pred)],
    'f1':       [f1_score(y_te, y_pred),       f1_score(y_te, log_pred)],
    'roc_auc':  [roc_auc_score(y_te, y_proba), roc_auc_score(y_te, log_proba)],
})
comparativa

💡 **Observa la diferencia.** En este dataset el árbol gana al lineal por mucho — señal clara de que hay **interacciones y no linealidades** (p. ej. `dtir1` y `LTV` actuando juntas) que la logística no logra capturar sin features manuales.

In [ ]:
# --- Curva de complejidad (max_depth) ---
depths = range(1, 16)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr.append(accuracy_score(y_tr, m.predict(X_tr)))
    te.append(accuracy_score(y_te, m.predict(X_te)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, tr, marker='o', label='Train accuracy')
ax.plot(depths, te, marker='o', label='Test  accuracy')
ax.set_xlabel('max_depth'); ax.set_ylabel('accuracy')
ax.set_title('Curva de complejidad — DecisionTreeClassifier')
ax.legend(); plt.show()

## 5. Referencias

- Breiman et al. (1984). *Classification and Regression Trees*.
- Quinlan, J. R. (1986). *Induction of Decision Trees*.
- ISLR cap. 8.
- scikit-learn user guide: https://scikit-learn.org/stable/modules/tree.html
- Dataset: https://www.kaggle.com/datasets/yasserh/loan-default-dataset

## Predicción sobre datos nuevos — uso del modelo en producción

Una vez que el modelo está validado en el conjunto de test, queremos usarlo para predecir sobre datos que **no hemos visto**. En la práctica seguimos tres pasos:

1. **Reentrenar con todos los datos disponibles.** Ya hicimos la validación con la partición train/test; ahora aprovechamos el 100% de la información para que el modelo final tenga la mejor estimación posible de los parámetros.
2. **Aplicar exactamente las mismas transformaciones** que durante el entrenamiento (mismas columnas, mismo encoding, misma escala) — un error muy común en producción es desalinear el preprocesamiento.
3. **Persistir el modelo** con `joblib` (o `pickle`) para reutilizarlo sin reentrenar.

In [ ]:
import joblib

# Reentrenamos con TODOS los datos
modelo_final = DecisionTreeClassifier(
    criterion='gini', max_depth=4, random_state=42,
).fit(X, y)

# Bundle: modelo + columnas + listas de num/cat para replicar el preprocesamiento
bundle = {
    'modelo':    modelo_final,
    'columnas':  X.columns.tolist(),
    'num_cols':  num_cols,
    'cat_cols':  cat_cols,
}
joblib.dump(bundle, 'modelo_loan_tree.pkl')
print('Bundle guardado.')

### Inferencia individual — una solicitud de préstamo nueva

Igual que en el notebook anterior, usamos `pd.get_dummies` + `reindex` para alinear columnas con el entrenamiento.

In [ ]:
nuevo_credito_raw = pd.DataFrame([{
    # numéricas
    'loan_amount':              250000,
    'rate_of_interest':         4.5,
    'term':                     360,    # meses
    'property_value':           350000,
    'income':                   7200,   # ingreso mensual
    'Credit_Score':             720,
    'LTV':                      71.0,   # loan-to-value (%)
    'dtir1':                    35.0,   # debt-to-income (%)
    # categóricas
    'loan_type':                'type1',
    'loan_purpose':             'p3',
    'Credit_Worthiness':        'l1',
    'business_or_commercial':   'nob/c',
    'occupancy_type':           'pr',
    'credit_type':              'EXP',
    'age':                      '35-44',
    'Region':                   'North',
    'Gender':                   'Male',
}])

nuevo = pd.get_dummies(nuevo_credito_raw, columns=cat_cols, drop_first=True)
nuevo = nuevo.reindex(columns=X.columns, fill_value=0)

prob_default = modelo_final.predict_proba(nuevo)[0, 1]
print(f'P(default) = {prob_default:.3f}')
print(f'Decisión @ umbral 0.5: {"DENEGAR" if prob_default >= 0.5 else "APROBAR"}')
print(f'Decisión @ umbral 0.3: {"DENEGAR" if prob_default >= 0.3 else "APROBAR"}  '
      f'(banco conservador)')

### Caso de uso: política de aprobación basada en riesgo

Un banco rara vez aprueba/rechaza con un umbral fijo. Lo típico es:

- `P(default) < 0.10` → **aprobación automática** con la tasa estándar.
- `0.10 ≤ P(default) < 0.30` → aprobar pero **subir la tasa** o pedir aval.
- `P(default) ≥ 0.30` → **revisión manual** o rechazo.

El modelo ordena las solicitudes; las reglas de negocio definen los umbrales.

In [ ]:
test_proba = modelo_final.predict_proba(X_te)[:, 1]

def politica(p):
    if p < 0.10:  return 'auto-aprobado'
    if p < 0.30:  return 'aprobado con condiciones'
    return 'revisión manual / rechazo'

decisiones = pd.DataFrame({
    'P_default': test_proba.round(3),
    'real':      y_te.values,
})
decisiones['politica'] = decisiones['P_default'].apply(politica)

print('Distribución de decisiones sobre el test set:')
print(decisiones['politica'].value_counts().to_frame('n'))

print('\nTasa de default REAL por categoría de la política:')
print(decisiones.groupby('politica')['real'].mean().round(3).to_frame('tasa_default'))

**Lecciones para producción:**
- En scoring crediticio el desbalance de costos es brutal: un falso negativo (prestar a alguien que va a default) puede costar miles de USD; un falso positivo (rechazar a un buen pagador) cuesta una venta perdida. **Ajustar el umbral** y **revisar la política** son tareas continuas, no decisiones de una sola vez.
- Los árboles dan reglas explicables — útil para cumplir con regulación que exige justificar la decisión al solicitante.
- En la práctica, un solo árbol se reemplaza por un **Random Forest** o **Gradient Boosting** (XGBoost / LightGBM): más AUC y aún explicable con SHAP.